In [1]:
import os
import json
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

def read_json(path):
    with open(path, 'r', encoding="utf-8") as f:
        data = json.load(f)
    return data

def write_json(data, path):
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [3]:
import numpy as np

def compute_forgetting_matrix(accuracy_matrix):
    """
    Computes forgetting for each task.
    
    Parameters:
    - accuracy_matrix: 2D numpy array of shape (num_tasks, num_tasks)
      Each row i represents accuracy on task i after training on tasks j=0..T-1
    
    Returns:
    - forgetting: list of forgetting values for each task
    - avg_forgetting: average forgetting across tasks (excluding the last task)
    """
    num_tasks = accuracy_matrix.shape[0]
    forgetting = []

    for i in range(num_tasks - 1):
        max_acc = np.max(accuracy_matrix[i, :i+1])  # Max accuracy before final training
        final_acc = accuracy_matrix[i, -1]          # Accuracy after training on all tasks
        forgetting.append(max_acc - final_acc)

    avg_forgetting = np.mean(forgetting)
    return forgetting, avg_forgetting


In [8]:
# matrices preparation on T5
def compute_forgetting_matrix_t5(input_folder, test_folder):
    results = []
    for i in range(1, 6):
       input_file = f"/Users/sefika/phd_projects/llm-catastrophic-re/results-finance/finance/results/llm_forget_ewc/{input_folder}/task_{i}_seen_task.json"
       start=0 
       predictions = read_json(input_file)
       predictions = [item['predict'] for item in predictions]
       
       for id in range(1, i+1):
        test_file = f"/Users/sefika/phd_projects/tgdk-paper/data/finance/data/cl_task/test/{test_folder}/task_{id}/test.json"
        test_data = read_json(test_file)
        y_true = [item['relation'] for item in test_data]
        y_task = predictions[start:start+len(y_true)]
        acc = accuracy_score(y_true, y_task)
        print(f"Task {i} - Task {id} Accuracy: {acc:.4f}")
        print(f"Task {i} - Task {id} Size: {len(y_task)}")
        print(f"Task {i} - Task {id} True Size: {len(y_true)}")

        row = {'base_task':i , 'task':id, 'accuracy':acc}
        
        start += len(y_true)
        results.append(row)
    return results
for run_id in range(1, 6):
    results = compute_forgetting_matrix_t5(
        input_folder=f"model_{run_id}",
        test_folder=f"run_{run_id}"
    )
    results
    write_json(results, f"/Users/sefika/phd_projects/llm-catastrophic-re/results-finance/finance/results/llm_forget_ewc/model_{run_id}_forgetting_matrix.json")


Task 1 - Task 1 Accuracy: 0.9817
Task 1 - Task 1 Size: 327
Task 1 - Task 1 True Size: 327
Task 2 - Task 1 Accuracy: 0.9817
Task 2 - Task 1 Size: 327
Task 2 - Task 1 True Size: 327
Task 2 - Task 2 Accuracy: 0.9905
Task 2 - Task 2 Size: 211
Task 2 - Task 2 True Size: 211
Task 3 - Task 1 Accuracy: 0.9817
Task 3 - Task 1 Size: 327
Task 3 - Task 1 True Size: 327
Task 3 - Task 2 Accuracy: 0.9810
Task 3 - Task 2 Size: 211
Task 3 - Task 2 True Size: 211
Task 3 - Task 3 Accuracy: 0.9188
Task 3 - Task 3 Size: 234
Task 3 - Task 3 True Size: 234
Task 4 - Task 1 Accuracy: 0.9817
Task 4 - Task 1 Size: 327
Task 4 - Task 1 True Size: 327
Task 4 - Task 2 Accuracy: 0.9858
Task 4 - Task 2 Size: 211
Task 4 - Task 2 True Size: 211
Task 4 - Task 3 Accuracy: 0.9060
Task 4 - Task 3 Size: 234
Task 4 - Task 3 True Size: 234
Task 4 - Task 4 Accuracy: 0.8727
Task 4 - Task 4 Size: 220
Task 4 - Task 4 True Size: 220
Task 5 - Task 1 Accuracy: 0.9633
Task 5 - Task 1 Size: 327
Task 5 - Task 1 True Size: 327
Task 5 - T

In [ ]:
# matrices preparation on Mistral

In [ ]:
# matrices preparation on Llama-2